In [ ]:
import os
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_HSE_HW')

In [ ]:
!gh repo clone https://github.com/Yaitco/NLP-HW2.git
%cd NLP-HW2

Первые две ячейки загружают токен из Colab Secrets и клонируют репозиторий. Для следующих ячеек нужны установленные зависимости и включённый GPU. Результат — `prediction.jsonl` в порядке test.

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from huggingface_hub import snapshot_download
from tokenizers import Tokenizer

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src.model import Seq2SeqTransformer, Seq2SeqTransformerConfig
from src.code_preprocessing import restore_code
from src.utils.code_postprocessing import restore_generated_code
from src.text_cleaning import clean_question
from src.utils.beam_voting import BeamVoter

REPO_ID = "YaItco/NLP-2-transformer"
DEVICE = "cuda"
BATCH_SIZE = 8
NUM_BEAMS = 16
LENGTH_PENALTY = 1.0
BEAM_VOTING = True
VOTING_TEMPERATURE = 1.0
SANDBOX = False  # Colab experiments: separate processes, without filesystem/network isolation.
EXECUTION_BATCH_SIZE = 16  # Arithmetic candidates per Python process.
MAX_NEW_TOKENS = 128
OUTPUT = "prediction.jsonl"

RUN_TUNING = False  # Enable optional validation cells below.
VALIDATION_PATH = "artifacts/code_noimports_full_clean/validation"
TUNING_CACHE = "validation_beams.json"

In [ ]:
folder = Path(snapshot_download(REPO_ID, allow_patterns=["model.pt", "config.json", "tokenizer.json"]))
config = json.loads((folder / "config.json").read_text())
model = Seq2SeqTransformer(Seq2SeqTransformerConfig(**config["model"]))
model.load_state_dict(torch.load(folder / "model.pt", map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()
tokenizer = Tokenizer.from_file(str(folder / "tokenizer.json"))
pad_id = config["model"]["pad_id"]
voter = BeamVoter(folder / "tokenizer.json", code_format=config["code_format"], sandbox=SANDBOX, execution_batch_size=EXECUTION_BATCH_SIZE, add_imports=config.get("add_imports", False), temperature=VOTING_TEMPERATURE) if BEAM_VOTING else None

Необязательный подбор параметров: поставь `RUN_TUNING = True` и укажи `VALIDATION_PATH` — папку validation из `artifacts/code_noimports_full_clean` (перенеси её в Colab/Kaggle). Нужны колонки question и code, где code — тело solve с result. Кандидаты и результаты сохраняются в `TUNING_CACHE`; повторный подбор не запускает модель или код.

Первая половина валидных эталонов — tune, вторая — check. Сначала выбираем alpha по лучшему исполняемому кандидату, затем temperature по голосованию при выбранном alpha. Check и full только для отчёта. Это пересчёт фиксированного пула beams: новая генерация с другим alpha может дать другой пул. Длина включает EOS; формула нашего генератора — `raw_score / length**alpha`.

In [ ]:
if RUN_TUNING:
    import math
    from concurrent.futures import ThreadPoolExecutor
    from fractions import Fraction
    from datasets import load_from_disk
    from src.utils.beam_voting import same_result, weighted_consensus_index

    validation = load_from_disk(VALIDATION_PATH)
    evaluator = BeamVoter(
        folder / "tokenizer.json", code_format=config["code_format"],
        sandbox=SANDBOX, execution_batch_size=EXECUTION_BATCH_SIZE,
        add_imports=config.get("add_imports", False),
    )
    cache_path = Path(TUNING_CACHE)
    cache_config = dict(
        model_revision=folder.name, model_config=config,
        validation_fingerprint=validation._fingerprint,
        num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS,
        length_penalty=LENGTH_PENALTY, sandbox=SANDBOX,
        return_last_assignment=True,
    )
    if cache_path.exists():
        cache = json.loads(cache_path.read_text())
        if cache["config"] != cache_config:
            raise ValueError("Cache settings differ: use a new TUNING_CACHE path or delete the old cache.")
    else:
        val_questions = list(validation["question"])
        if config.get("clean_questions", False):
            val_questions = [clean_question(q) for q in val_questions]
        val_encoded = tokenizer.encode_batch(val_questions, add_special_tokens=False)
        records = [None] * len(validation)
        val_order = sorted(range(len(records)), key=lambda i: len(val_encoded[i].ids))
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            for offset in range(0, len(val_order), BATCH_SIZE):
                batch_indices = val_order[offset:offset + BATCH_SIZE]
                rows = [torch.tensor(val_encoded[i].ids, dtype=torch.long) for i in batch_indices]
                ids = pad_sequence(rows, batch_first=True, padding_value=pad_id).to(DEVICE)
                beams, scores = model.generate(
                    ids, attention_mask=ids.ne(pad_id), num_beams=NUM_BEAMS,
                    max_new_tokens=MAX_NEW_TOKENS, length_penalty=LENGTH_PENALTY,
                    use_cache=True, return_beams=True,
                )
                details = evaluator.evaluate(beams, scores)
                for b, i in enumerate(batch_indices):
                    records[i] = dict(
                        codes=details["codes"][b],
                        results=[str(v) if v is not None else None for v in details["results"][b]],
                        lengths=details["lengths"][b],
                        scores=[v if math.isfinite(v) else None for v in details["scores"][b]],
                    )
                del ids, beams, scores
        reference_codes = [
            restore_code(code, add_imports=config.get("add_imports", False))
            if config["code_format"] == "solve_body_result" else code
            for code in validation["code"]
        ]
        with ThreadPoolExecutor(max_workers=evaluator.workers) as pool:
            reference_results = list(pool.map(evaluator.executor.run, reference_codes))
        cache = dict(config=cache_config, records=records, references=[
            str(r.value) if r.status == "ok" else None for r in reference_results
        ])
        cache_path.write_text(json.dumps(cache, ensure_ascii=False, allow_nan=False))
    all_codes = [r["codes"] for r in cache["records"]]
    all_results = [[Fraction(v) if v is not None else None for v in r["results"]]
                   for r in cache["records"]]
    all_scores = [[v if v is not None else -math.inf for v in r["scores"]]
                  for r in cache["records"]]
    all_lengths = [r["lengths"] for r in cache["records"]]
    references = [Fraction(v) if v is not None else None for v in cache["references"]]
    print("Cached questions:", len(references), "reference errors:", references.count(None))


In [ ]:
if RUN_TUNING:
    from src.utils.beam_voting import rescore_lengths

    indices = [i for i, ref in enumerate(references) if ref is not None]
    if len(indices) < 2:
        raise ValueError("Need at least two valid references for tune/check.")
    middle = len(indices) // 2
    tune, check = indices[:middle], indices[middle:]
    source_penalty = cache["config"]["length_penalty"]
    alphas = [x / 10 for x in range(16)]  # generate() supports non-negative alpha.
    temperatures = [0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]

    def scores_for(alpha):
        return [rescore_lengths(s, lengths, source_penalty, alpha)
                for s, lengths in zip(all_scores, all_lengths)]

    def candidate_index(i, scores):
        order = sorted(range(len(scores[i])), key=lambda j: scores[i][j], reverse=True)
        return next((j for j in order if all_results[i][j] is not None
                     and math.isfinite(scores[i][j])), order[0])

    def accuracy(subset, scores, temperature=None):
        correct = 0
        for i in subset:
            j = (candidate_index(i, scores) if temperature is None else
                 weighted_consensus_index(all_results[i], scores[i], temperature))
            correct += same_result(all_results[i][j], references[i])
        return correct / len(subset)

    best_alpha = max(alphas, key=lambda a: accuracy(tune, scores_for(a)))
    rescored = scores_for(best_alpha)
    best_temperature = max(temperatures, key=lambda t: accuracy(tune, rescored, t))
    print("length_penalty:", best_alpha, "temperature:", best_temperature)
    for name, subset in (("tune", tune), ("check", check), ("full", indices)):
        print(name, "best executable:", accuracy(subset, rescored),
              "weighted:", accuracy(subset, rescored, best_temperature))

    LENGTH_PENALTY = best_alpha
    VOTING_TEMPERATURE = best_temperature
    if voter is not None:
        voter.temperature = best_temperature
    print("Selected parameters applied to test generation below.")


In [ ]:
test = load_dataset("SlavaYus/tinysms_qa_12m", split="test", streaming=True)
questions = [row["question"] for row in test]

In [ ]:
torch.cuda.synchronize()
start = perf_counter()
inputs = [clean_question(q) for q in questions] if config.get("clean_questions", False) else questions
encoded = tokenizer.encode_batch(inputs, add_special_tokens=False)
order = sorted(range(len(questions)), key=lambda i: len(encoded[i].ids))
predictions = [None] * len(questions)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    for offset in range(0, len(order), BATCH_SIZE):
        indices = order[offset:offset + BATCH_SIZE]
        rows = [torch.tensor(encoded[i].ids, dtype=torch.long) for i in indices]
        input_ids = pad_sequence(rows, batch_first=True, padding_value=pad_id).to(DEVICE)
        generated = model.generate(
            input_ids, attention_mask=input_ids.ne(pad_id),
            max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, use_cache=True,
            length_penalty=LENGTH_PENALTY,
            return_beams=BEAM_VOTING,
        )
        if voter is not None:
            generated = voter.select(*generated)
        codes = tokenizer.decode_batch(generated.cpu().tolist(), skip_special_tokens=True)
        for i, code in zip(indices, codes):
            predictions[i] = restore_generated_code(code, add_imports=config.get("add_imports", False)) if config["code_format"] == "solve_body_result" else code

torch.cuda.synchronize()
print(f"{len(predictions)} examples, {perf_counter() - start:.1f} s (tokenization + generation + voting)")

In [ ]:
with open(OUTPUT, "w", encoding="utf-8") as file:
    for code in predictions:
        file.write(json.dumps({"code": code}, ensure_ascii=False) + "\n")